# IEVF Agreement-Bias Study - Colab GPU runner

**Before running:**
1. Upload the whole `research_ievf` folder to Google Drive (e.g. `MyDrive/research_ievf`).
2. Runtime -> Change runtime type -> **T4 GPU**.
3. Run the cells top to bottom. Every phase is **resume-safe**: if the runtime
   disconnects, reconnect and re-run the same cell - finished work is skipped.

All models run **locally via Hugging Face transformers on the Colab GPU - no API
keys, no cost**. Study models (4, all different families): SmolLM2-1.7B, Qwen2.5-1.5B,
Qwen2.5-3B, Phi-3.5-mini. Judge: Qwen2.5-3B under family `hf-judge`, so it never
shares a family with the model it grades (independence rule). Only 2 models are
kept in VRAM at a time (study model + judge), the rest are cycled in/out.

In [6]:
# 0. Mount Drive and copy the project to the Colab VM (fast local IO)
from google.colab import drive
drive.mount('/content/drive')

# EDIT THIS if you put the folder somewhere else in Drive
DRIVE_PATH = "/content/drive/MyDrive/research_ievf"

!rm -rf /content/research_ievf
!cp -r "$DRIVE_PATH" /content/research_ievf
%cd /content/research_ievf
!ls

# True = ONLY the local Hugging Face models run (no API calls at
# all, even if your .env holds Google/Groq/OpenAI keys).
LOCAL_ONLY = True
LOCAL_MODELS = "hf-smollm,hf-qwen,hf-qwen3b"
MODELS_FLAG = f" --models {LOCAL_MODELS}" if LOCAL_ONLY else ""

# OPTIONAL: keep model weights on Drive so they survive a
# 'Factory reset runtime' (otherwise Colab re-downloads them,
# ~1-2 min total). Safe to leave True.
HF_CACHE_ON_DRIVE = True
if HF_CACHE_ON_DRIVE:
    !mkdir -p /root/.cache/huggingface
    !mkdir -p "$DRIVE_PATH/hf_cache"
    !cp -rn "$DRIVE_PATH/hf_cache/." /root/.cache/huggingface/ 2>/dev/null || true


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
/content/research_ievf
benchmark	 config.yaml  logs			reports		  src
check_setup.py	 data	      make_experiment_guide.py	requirements.txt
colab_run.ipynb  hf_cache     README.md			run_pipeline.py


In [7]:
# 1. Dependencies (Colab already has torch/transformers; this only fills gaps)
!pip install -q -r requirements.txt

# GPU check - you should see a Tesla T4
!nvidia-smi

Sat Aug 15 17:40:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P0             29W /   70W |   12987MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
# 2. Preflight: shows every model, its judge, the question bank and the DB.
# All hf-* models must show 'OK ... [local]' with no token needed.
!python check_setup.py

--- models (config.yaml + .env) ---
--  gpt-5           gpt-5                        family=openai     token=OPENAI_API_KEY (missing)
--  claude-sonnet   claude-sonnet-4-5-20250929   family=anthropic  token=ANTHROPIC_API_KEY (missing)
OK  gemini          gemini-flash-latest          family=google     token=GOOGLE_API_KEY
--  deepseek        deepseek-chat                family=deepseek   token=DEEPSEEK_API_KEY (missing)
--  qwen            qwen-plus                    family=alibaba    token=DASHSCOPE_API_KEY (missing)
--  kimi            moonshot-v1-8k               family=moonshot   token=MOONSHOT_API_KEY (missing)
OK  groq            llama-3.3-70b-versatile      family=groq       token=GROQ_API_KEY
OK  hf-smollm       HuggingFaceTB/SmolLM2-1.7B-Instruct family=smollm     token=(none - runs locally) [local]
OK  hf-qwen         Qwen/Qwen2.5-1.5B-Instruct   family=qwen       token=(none - runs locally) [local]
OK  hf-qwen3b       Qwen/Qwen2.5-3B-Instruct     family=qwen       token=(non

In [9]:
# 3. Build the question bank (MoReBench + SimpleBench + AschBench -> 510 items)
!python run_pipeline.py --phase fetch-data --simplebench --aschbench

AschBench: 100 items from 1 file(s), 0 format errors
Saved 510 items -> /content/research_ievf/data/processed/items.jsonl


In [10]:
# EXPERIMENTS (no mitigation): baseline -> exp1 -> exp2, all in ONE
# process so the models stay in VRAM. Resume-safe: finished combinations
# are skipped. After this, run the ANALYSIS cell to see whether bias
# exists. Only then decide whether to run the MITIGATION cell below.
LIMIT = 5          # set to None for the full 510-item study

from src import data_loader, models, pipeline

# build the question bank if it is missing
_bank = data_loader.ROOT / "data" / "processed" / "items.jsonl"
if not _bank.exists():
    print("[info] question bank missing - building it now")
    _items = data_loader.load_morebench(n=250, include_theory=True)
    _items += data_loader.load_simplebench()
    _items += data_loader.load_aschbench()
    data_loader.save_items(_items)

items = data_loader.load_items()
keys = (models.study_models(LOCAL_MODELS.split(",")) if LOCAL_ONLY
        else models.study_models())
print("running:", keys)

for phase, mit in [("baseline", False),
                   ("exp1", False), ("exp2", False)]:
    pipeline.run(items, keys, phase, limit=LIMIT, mit=mit)

# sync any newly downloaded weights back to Drive
if HF_CACHE_ON_DRIVE:
    !cp -rn /root/.cache/huggingface/. "$DRIVE_PATH/hf_cache/" 2>/dev/null || true


running: ['hf-smollm', 'hf-qwen', 'hf-qwen3b']
baseline: hf-smollm graded by judge-hf-qwen


baseline/hf-smollm: 100%|██████████| 5/5 [01:21<00:00, 16.30s/it]


baseline: hf-qwen graded by judge-hf-qwen


baseline/hf-qwen: 100%|██████████| 5/5 [01:23<00:00, 16.64s/it]


baseline: hf-qwen3b graded by judge-hf-qwen


...

exp2/hf-smollm: 100%|██████████| 5/5 [18:05<00:00, 217.14s/it]


exp2: hf-qwen graded by judge-hf-qwen


exp2/hf-qwen: 100%|██████████| 5/5 [23:28<00:00, 281.63s/it]


exp2: hf-qwen3b graded by judge-hf-qwen


exp2/hf-qwen3b: 100%|██████████| 5/5 [42:21<00:00, 508.33s/it]


In [15]:
# ANALYSIS 1 - does bias exist? Reads results.db, writes tables + figures
# to reports/. Look at metrics_by_condition.csv (drop_rate per P-cell and
# group size) and dose_response.png: rising drop_rate = conformity.
!python run_pipeline.py --phase analyze
!ls -la reports reports/figures


Reports written to /content/research_ievf/reports
reports:
total 52
drwx------  3 root root  4096 Aug 15 19:54 .
drwx------ 10 root root  4096 Aug 15 17:39 ..
-rw-------  1 root root 10519 Aug 15 17:39 experiment_guide.pdf
drwx------  2 root root  4096 Aug 15 19:54 figures
-rw-r--r--  1 root root   145 Aug 15 19:56 hysteresis.csv
-rw-r--r--  1 root root   141 Aug 15 19:56 metrics_by_benchmark.csv
-rw-r--r--  1 root root  1022 Aug 15 19:56 metrics_by_condition.csv
-rw-r--r--  1 root root   301 Aug 15 19:56 metrics_by_model.csv
-rw-------  1 root root  5006 Aug 15 17:39 report_1_pipeline.pdf
-rw-------  1 root root  3936 Aug 15 17:39 report_2_experiment_flow.pdf

reports/figures:
total 124
drwx------ 2 root root  4096 Aug 15 19:54 .
drwx------ 3 root root  4096 Aug 15 19:54 ..
-rw------- 1 root root 23784 Aug 15 17:39 before_after.png
-rw------- 1 root root 58917 Aug 15 19:56 dose_response.png
-rw-r--r-- 1 root root 29495 Aug 15 19:56 per_model.png


In [12]:
# # MITIGATION (optional - run only if the analysis above shows bias):
# # reruns exp1 and exp2 with the IEVF+EGDA gate switched on. Results are
# # stored separately as exp1_mit / exp2_mit so before/after stay
# # comparable. No bias found? Simply never run this cell.
# # Models are still in VRAM from the experiments cell -> no reloads.
# LIMIT = 30          # keep the SAME limit as the experiments cell

# from src import data_loader, models, pipeline

# # build the question bank if it is missing
# _bank = data_loader.ROOT / "data" / "processed" / "items.jsonl"
# if not _bank.exists():
#     print("[info] question bank missing - building it now")
#     _items = data_loader.load_morebench(n=250, include_theory=True)
#     _items += data_loader.load_simplebench()
#     _items += data_loader.load_aschbench()
#     data_loader.save_items(_items)

# items = data_loader.load_items()
# keys = (models.study_models(LOCAL_MODELS.split(",")) if LOCAL_ONLY
#         else models.study_models())
# print("running:", keys)

# for phase in ("exp1", "exp2"):
#     pipeline.run(items, keys, phase, limit=LIMIT, mit=True)

# if HF_CACHE_ON_DRIVE:
#     !cp -rn /root/.cache/huggingface/. "$DRIVE_PATH/hf_cache/" 2>/dev/null || true


In [13]:
# # ANALYSIS 2 - before vs after IEVF. Rerun ONLY after the mitigation
# # cell. before_after.csv / before_after.png now compare drop rates with
# # the gate off vs on. (before_after.csv exists only if *_mit data is in
# # the database.)
# !python run_pipeline.py --phase analyze
# !ls -la reports reports/figures


In [14]:
# 10. Copy results back to Drive (database + reports; the data/processed
# bank too, so you don't rebuild it next session).
!mkdir -p "$DRIVE_PATH/logs" "$DRIVE_PATH/reports" "$DRIVE_PATH/data"
!cp -r logs/. "$DRIVE_PATH/logs/"
!cp -r reports/. "$DRIVE_PATH/reports/"
!cp -r data/processed "$DRIVE_PATH/data/" 2>/dev/null || true
print('Saved back to', DRIVE_PATH)

Saved back to /content/drive/MyDrive/research_ievf


## Notes
- **Resume-safe**: every cell can be interrupted and re-run; completed
  (phase, model, item, condition, round) combinations are skipped.
- **Scale up**: remove `--limit` once the pilot looks right. For the full
  510-item study consider running one phase per Colab session.
- **More/fewer models**: edit `models:` in `config.yaml`. Any
  `provider: huggingface` entry runs locally with no token. Keep `family`
  unique per model lineage - the judge must never share the target's family.
- **Gated models** (Llama etc.): accept the license on huggingface.co, put
  `HF_TOKEN=...` in `.env` (or `import os; os.environ['HF_TOKEN']='hf_...'`
  in a cell before the runs), then uncomment the entry in `config.yaml`.
- **Out of VRAM**: the heaviest pair is Phi-3.5-mini + judge. If you see
  CUDA OOM, comment out `hf-phi` in `config.yaml`.